In [1]:
import pandas as pd

df = pd.read_csv("../Data/reviews_final.csv")
print(df.shape)

(34625, 12)


In [2]:
df["negative"] = (df["sentiment"] == "negative") * 100
df["recommend"] = df["do_recommend"] * 100

In [3]:
df["short_name"] = df["product_name"]
df["n_words"] = df["clean_text"].str.split().str.len()

print(df["category"].value_counts())

category
Tablets                  17596
Echo & Smart Speakers     7262
Fire TV                   5068
Kindle                    4137
Chargers, Cables           562
Name: count, dtype: int64


In [4]:
MIN_REVIEWS = 10

def product_table(category_df):
    table = category_df.groupby("short_name").agg(
        reviews=("rating", "size"),
        avg_rating=("rating", "mean"),
        pct_negative=("negative", "mean"),
        pct_recommend=("recommend", "mean"),
    ).round(2)
    table = table[table["reviews"] >= MIN_REVIEWS]
    return table.sort_values("avg_rating", ascending=False)

product_table(df[df["category"] == "Chargers, Cables"])

,reviews,avg_rating,pct_negative,pct_recommend
short_name,,,,
amazon echo fire tv power adapter,16,4.50,6.25,NaN
amazon 5 w usb official oem charger power adapter for fire tablets kind lee readers,416,4.39,10.34,NaN
amazon 9 w power fast official oem usb charger power adapter for fire tablets kind lee readers,73,4.21,16.44,94.44
amazon fire hd 6 standing protective case 4 th generation 2014 release cayenne red,13,3.92,23.08,NaN
kindled x leather cover black fits 97 display latest 2 nd generation kindled xs,10,3.50,30.00,NaN
new amazon kindle fire hd 9 w power fast adapter charger micro usb angle cable,13,2.46,61.54,NaN


In [5]:
def complaints(category_df, product, n=2):
    unhappy = category_df[(category_df["short_name"] == product) & (category_df["rating"] <= 3) & (category_df["n_words"] >= 10)]
    examples = unhappy["clean_text"].sample(min(n, len(unhappy)), random_state=42)
    return [text[:150] for text in examples]

def pick_products(table):
    if len(table) < 2:
        return list(table.index[:3]), None
    worst = table["avg_rating"].idxmin()
    top = table.drop(worst).sort_values("reviews", ascending=False).index[:3]
    return list(top), worst

In [6]:
def product_line(table, product):
    row = table.loc[product]
    line = f"{product}: {row['avg_rating']}/5 from {row['reviews']:.0f} reviews, {row['pct_negative']}% negative"
    if pd.notna(row["pct_recommend"]):
        line += f", {row['pct_recommend']}% recommend it"
    return line

def facts_text(category):
    category_df = df[df["category"] == category]
    table = product_table(category_df)
    top, worst = pick_products(table)

    lines = [f"Category: {category} ({len(category_df)} reviews)", "", "TOP PRODUCTS (most reviewed first):"]
    for product in top:
        lines.append("- " + product_line(table, product))
        for text in complaints(category_df, product):
            lines.append(f'    complaint: "{text}"')
    lines.append("")
    if worst is None:
        lines.append("WORST PRODUCT: none")
    else:
        lines.append("WORST PRODUCT: " + product_line(table, worst))
        for text in complaints(category_df, worst):
            lines.append(f'    complaint: "{text}"')
    return "\n".join(lines)

categories = sorted(df["category"].unique())
print(categories)
print(facts_text("Chargers, Cables"))

['Chargers, Cables', 'Echo & Smart Speakers', 'Fire TV', 'Kindle', 'Tablets']
Category: Chargers, Cables (562 reviews)

TOP PRODUCTS (most reviewed first):
- amazon 5 w usb official oem charger power adapter for fire tablets kind lee readers: 4.39/5 from 416 reviews, 10.34% negative
    complaint: "a waste of money . a waste of money. you don't need this. just plug your tablet into your computer and the charge lasts for a month"
    complaint: "only lasts for less than 1 year! . this is the third power charger that i have had to buy at cost of over 30 each since my kindle fire purchase. why c"
- amazon 9 w power fast official oem usb charger power adapter for fire tablets kind lee readers: 4.21/5 from 73 reviews, 16.44% negative, 94.44% recommend it
    complaint: "defective. . worked wonderful and fast the first couple of weeks and then just stopped working. i understand that sometimes things are just a defectiv"
    complaint: "power fast how about power not! . this doesn't power any

In [ ]:
import torch
from transformers import pipeline

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

writer = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.bfloat16, device=device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\PCACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PCACER\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            